# Train Mask2Former on the nanostar data (Kaggle, self-contained)

This notebook is **self-contained** — all the dataset / augmentation / training code
is inlined, so you only upload the data. It reproduces the repo's pipeline:

* **Two script-free square crops per image** — `A = [0:3800, 0:3800]` and
  `B = [900:4096, 900:4096]`. Both avoid the burned-in *"100 nm"* scale-bar in the
  bottom-left (bbox ≈ `(46, 3804, 900, 4049)` on 4096² frames) and between them
  recover almost the whole frame. Every image is enumerated as **both** crops.
* **Augmentation = flip + rotation at every 30°.** 90° multiples are exact
  transposes; 30/60/120/... are *rotate-then-inscribe* (rotate about the centre,
  crop the largest upright square that still fits) so the patch never leaves the
  crop — no black padding. Masks rotate with the image and stay binary.
* **Training on crops, validation on the full 4096² frame** (matches real
  inference). Each epoch reports **val loss** and **COCO mask AP**.

> Note: the repo's optional mask-denoising branch (`src/m2f_denoise.py`) is **not**
> included here — it reimplements Mask2Former decoder internals against a pinned
> `transformers` version. This notebook trains a plain Mask2Former, which runs on
> current `transformers`. Use the repo scripts if you want denoising.

## 1 · Upload the data as a Kaggle Dataset

The images are large (407 × 4096² JPEGs, ≈4 GB), so upload them as a Kaggle Dataset
(**Datasets → New Dataset**). The notebook auto-detects the layout under
`/kaggle/input`, but the simplest structure is:

```
<your-dataset>/
  images/        0.jpg, 1.jpg, ...          (the 407 frames)
  annotations/
    train.json   (COCO instances)
    valid.json   (COCO instances)
```

Then: **Code → New Notebook → Add Data** (attach the dataset) → **Settings:
Accelerator = GPU (T4/P100), Internet = On** (so `from_pretrained` can fetch the
backbone) → **File → Upload Notebook** (this file) → **Run All**.

Keep `SMOKE_TEST = True` for the first run (1 epoch / few samples) to prove it works
end-to-end, then set it `False` and Run All again for the real training.

In [ ]:
# --- dependencies -------------------------------------------------------------
# Kaggle ships torch / opencv / pycocotools / scipy. We only (re)install a recent
# transformers that supports the Mask2Former image processor square-resize path.
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "-q", "install",
                "transformers>=4.45", "pycocotools", "scipy"], check=False)
import transformers; print("transformers", transformers.__version__)

In [ ]:
# --- config -------------------------------------------------------------------
import glob
from pathlib import Path

MODEL             = "facebook/mask2former-swin-tiny-coco-instance"
IMG_SIZE          = 1024     # full crop is downscaled to IMG_SIZE² (no tiling)
BATCH_SIZE        = 1        # 1024² on swin-tiny fits a 16 GB T4 at batch 1
GRAD_ACCUM        = 2        # effective batch = BATCH_SIZE * GRAD_ACCUM
EPOCHS            = 20
LR                = 5e-5
SAMPLES_PER_EPOCH = 600
NUM_WORKERS       = 2
MIN_PIXELS        = 256      # drop instances smaller than this after cropping
SCORE_THRESH      = 0.5      # confidence threshold for AP eval
EVAL_EVERY        = 1        # run validation every N epochs
SMOKE_TEST        = True     # first run: 1 epoch, few samples/eval images
OUT_DIR           = Path("/kaggle/working/checkpoints/mask2former-nanostar")

# --- auto-detect the uploaded data under /kaggle/input ------------------------
def _find(*names):
    for root in ("/kaggle/input", "."):
        for n in names:
            hits = sorted(glob.glob(f"{root}/**/{n}", recursive=True))
            if hits:
                return hits[0]
    return None

TRAIN_JSON = _find("train.json")
VALID_JSON = _find("valid.json", "eval.json")
_sample    = _find("0.jpg", "2.jpg", "1.jpg")
IMAGES_DIR = str(Path(_sample).parent) if _sample else _find("images")

assert TRAIN_JSON and VALID_JSON and IMAGES_DIR, (
    "Could not locate data. Expected train.json / valid.json and the image folder "
    f"under /kaggle/input. Found: train={TRAIN_JSON} valid={VALID_JSON} images={IMAGES_DIR}")
print("TRAIN_JSON:", TRAIN_JSON)
print("VALID_JSON:", VALID_JSON)
print("IMAGES_DIR:", IMAGES_DIR)

In [ ]:
# --- imports ------------------------------------------------------------------
import json, math, random
import cv2, numpy as np, torch
from PIL import Image, ImageDraw
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from transformers import (Mask2FormerForUniversalSegmentation,
                          Mask2FormerImageProcessor)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

In [ ]:
# --- geometry, crops, augmentation (inlined from src/) ------------------------
def polygons_to_mask(polygons, h, w):
    m = Image.new("L", (w, h), 0)
    d = ImageDraw.Draw(m)
    for poly in polygons:
        if len(poly) >= 6:
            d.polygon([(poly[i], poly[i + 1]) for i in range(0, len(poly), 2)], fill=1)
    return np.asarray(m, dtype=np.uint8)

# Two script-free square crops for 4096² frames (scale-bar bbox ≈ (46,3804,900,4049)).
BAR_FREE_CROPS = ((0, 0, 3800, 3800), (900, 900, 4096, 4096))
ROT_ANGLES = tuple(range(0, 360, 30))  # 0,30,...,330

def _crop_polys(seg, x0, y0):
    """Shift COCO polygons into a crop's local frame (origin -> x0, y0)."""
    return [[c - (x0 if i % 2 == 0 else y0) for i, c in enumerate(poly)] for poly in seg]

def _inscribed_side(angle, side):
    """Largest upright square that fits inside side×side after rotating by angle."""
    t = math.radians(angle)
    return int(round(side / (abs(math.cos(t)) + abs(math.sin(t)))))

def _flip_rot(img, masks, side):
    """Random h-flip + rotation drawn from ROT_ANGLES on a square crop and its
    (N, side, side) masks. 90° multiples are exact transposes; 30/60/... rotate
    about the centre and crop the inscribed square (no padding). Image and masks
    share one affine; masks rotate nearest-neighbour to stay binary."""
    arr = np.ascontiguousarray(np.asarray(img))
    if random.random() < 0.5:
        arr = np.ascontiguousarray(arr[:, ::-1])
        if len(masks):
            masks = np.ascontiguousarray(masks[:, :, ::-1])
    angle = random.choice(ROT_ANGLES)
    if angle % 90 == 0:
        k = angle // 90
        if k:
            arr = np.ascontiguousarray(np.rot90(arr, k=k, axes=(0, 1)))
            if len(masks):
                masks = np.ascontiguousarray(np.rot90(masks, k=k, axes=(1, 2)))
    else:
        c = side / 2.0
        M = cv2.getRotationMatrix2D((c, c), angle, 1.0)
        arr = cv2.warpAffine(arr, M, (side, side), flags=cv2.INTER_LINEAR)
        if len(masks):
            masks = np.stack(
                [cv2.warpAffine(m, M, (side, side), flags=cv2.INTER_NEAREST) for m in masks],
                axis=0)
        s = _inscribed_side(angle, side)
        o = (side - s) // 2
        arr = np.ascontiguousarray(arr[o:o + s, o:o + s])
        if len(masks):
            masks = np.ascontiguousarray(masks[:, o:o + s, o:o + s])
    return Image.fromarray(arr), masks

In [ ]:
# --- dataset ------------------------------------------------------------------
class Mask2FormerDataset(Dataset):
    """One sample = one of the two script-free crops of a frame. `augment=False`
    yields the deterministic, un-rotated crops (used for validation loss)."""
    def __init__(self, coco_json, images_dir, processor, size=1024,
                 samples_per_epoch=None, min_pixels=256, augment=True):
        with open(coco_json) as f:
            coco = json.load(f)
        self.images = list(coco["images"])
        self.images_dir = Path(images_dir)
        self.processor = processor
        self.min_pixels = min_pixels
        self.augment = augment
        self.anns_by_img = {}
        for a in coco["annotations"]:
            if a.get("segmentation"):
                self.anns_by_img.setdefault(a["image_id"], []).append(a)
        self.samples = [(info, box) for info in self.images for box in BAR_FREE_CROPS]
        self.samples_per_epoch = samples_per_epoch or len(self.samples)

    def __len__(self):
        return self.samples_per_epoch

    def __getitem__(self, idx):
        info, (x0, y0, x1, y1) = self.samples[idx % len(self.samples)]
        img = Image.open(self.images_dir / info["file_name"]).convert("RGB").crop((x0, y0, x1, y1))
        W, H = img.size  # square: W == H
        anns = self.anns_by_img.get(info["id"], [])
        if anns:
            masks = np.stack(
                [polygons_to_mask(_crop_polys(a["segmentation"], x0, y0), H, W) for a in anns], axis=0)
        else:
            masks = np.zeros((0, H, W), dtype=np.uint8)

        if self.augment:
            img, masks = _flip_rot(img, masks, side=W)

        if len(masks):
            keep = masks.reshape(len(masks), -1).sum(axis=1) >= self.min_pixels
            masks = masks[keep]
        if len(masks) == 0:
            return self.__getitem__(random.randrange(len(self.samples)))

        Hc, Wc = masks.shape[1:]
        seg_map = np.zeros((Hc, Wc), dtype=np.uint16)
        inst2sem = {}
        for i, m in enumerate(masks, start=1):
            seg_map[m > 0] = i
            inst2sem[i] = 0  # single class
        enc = self.processor(images=[img], segmentation_maps=[seg_map],
                             instance_id_to_semantic_id=inst2sem,
                             ignore_index=0, return_tensors="pt")
        return {"pixel_values": enc["pixel_values"][0],
                "pixel_mask": enc["pixel_mask"][0],
                "mask_labels": enc["mask_labels"][0],
                "class_labels": enc["class_labels"][0]}

def m2f_collate(batch):
    return {"pixel_values": torch.stack([b["pixel_values"] for b in batch]),
            "pixel_mask": torch.stack([b["pixel_mask"] for b in batch]),
            "mask_labels": [b["mask_labels"] for b in batch],
            "class_labels": [b["class_labels"] for b in batch]}

In [ ]:
# --- processor, model, data loaders -------------------------------------------
processor = Mask2FormerImageProcessor.from_pretrained(MODEL)
processor.size = {"height": IMG_SIZE, "width": IMG_SIZE}  # exact square resize
processor.do_resize = True

model = Mask2FormerForUniversalSegmentation.from_pretrained(
    MODEL, id2label={0: "nanostar"}, label2id={"nanostar": 0},
    ignore_mismatched_sizes=True).to(device)

if SMOKE_TEST:
    EPOCHS, SAMPLES_PER_EPOCH = 1, 8

train_ds = Mask2FormerDataset(TRAIN_JSON, IMAGES_DIR, processor, size=IMG_SIZE,
                              samples_per_epoch=SAMPLES_PER_EPOCH, min_pixels=MIN_PIXELS,
                              augment=True)
val_ds   = Mask2FormerDataset(VALID_JSON, IMAGES_DIR, processor, size=IMG_SIZE,
                              min_pixels=MIN_PIXELS, augment=False)  # deterministic crops
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                      num_workers=NUM_WORKERS, collate_fn=m2f_collate,
                      pin_memory=device == "cuda")
val_dl   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=NUM_WORKERS, collate_fn=m2f_collate)
print(f"train crops/epoch: {len(train_ds)}   val crops: {len(val_ds)}")

In [ ]:
# --- validation: val loss (on crops) + COCO mask AP (on full frames) ----------
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from pycocotools import mask as mask_utils

@torch.no_grad()
def val_loss(model, loader):
    model.eval()
    tot, n = 0.0, 0
    for batch in loader:
        out = model(pixel_values=batch["pixel_values"].to(device),
                    pixel_mask=batch["pixel_mask"].to(device),
                    mask_labels=[m.to(device) for m in batch["mask_labels"]],
                    class_labels=[c.to(device) for c in batch["class_labels"]])
        tot += float(out.loss); n += 1
    return tot / max(n, 1)

@torch.no_grad()
def eval_mask_ap(model, coco_json, images_dir, max_images=None):
    """Run the model on FULL 4096² frames (matches inference) and score COCO segm AP."""
    model.eval()
    coco_gt = COCO(coco_json)
    img_ids = coco_gt.getImgIds()
    if max_images:
        img_ids = img_ids[:max_images]
    results = []
    for img_id in tqdm(img_ids, desc="eval", leave=False):
        info = coco_gt.loadImgs(img_id)[0]
        img = Image.open(Path(images_dir) / info["file_name"]).convert("RGB")  # full frame
        enc = processor(images=img, return_tensors="pt").to(device)
        out = model(**enc)
        res = processor.post_process_instance_segmentation(
            out, target_sizes=[(info["height"], info["width"])], threshold=SCORE_THRESH)[0]
        seg = res["segmentation"]
        if seg is None:
            continue
        seg = seg.cpu().numpy()
        for s in res["segments_info"]:
            m = np.asfortranarray((seg == s["id"]).astype(np.uint8))
            rle = mask_utils.encode(m)
            rle["counts"] = rle["counts"].decode()
            results.append({"image_id": img_id, "category_id": 1,
                            "segmentation": rle, "score": float(s["score"])})
    if not results:
        return {"mAP": 0.0, "mAP50": 0.0, "n_pred": 0}
    coco_dt = coco_gt.loadRes(results)
    ev = COCOeval(coco_gt, coco_dt, "segm")
    if max_images:
        ev.params.imgIds = img_ids
    ev.evaluate(); ev.accumulate(); ev.summarize()
    return {"mAP": float(ev.stats[0]), "mAP50": float(ev.stats[1]), "n_pred": len(results)}

In [ ]:
# --- train --------------------------------------------------------------------
opt = torch.optim.AdamW(model.parameters(), lr=LR)
OUT_DIR.mkdir(parents=True, exist_ok=True)
eval_cap = 4 if SMOKE_TEST else None  # smoke: score only a few frames

for epoch in range(EPOCHS):
    model.train()
    running, opt_steps = 0.0, 0
    opt.zero_grad()
    pbar = tqdm(train_dl, desc=f"epoch {epoch + 1}/{EPOCHS}")
    for step, batch in enumerate(pbar):
        out = model(pixel_values=batch["pixel_values"].to(device),
                    pixel_mask=batch["pixel_mask"].to(device),
                    mask_labels=[m.to(device) for m in batch["mask_labels"]],
                    class_labels=[c.to(device) for c in batch["class_labels"]])
        loss = out.loss / GRAD_ACCUM
        loss.backward()
        if (step + 1) % GRAD_ACCUM == 0:
            opt.step(); opt.zero_grad(); opt_steps += 1
        running += float(out.loss)
        pbar.set_postfix(loss=f"{float(out.loss):.4f}")
    train_loss = running / max(len(train_dl), 1)

    # save a plain checkpoint every epoch (resumable / downloadable)
    model.save_pretrained(OUT_DIR); processor.save_pretrained(OUT_DIR)

    line = f"epoch {epoch + 1}: train_loss {train_loss:.4f}"
    if (epoch + 1) % EVAL_EVERY == 0:
        vl = val_loss(model, val_dl)
        ap = eval_mask_ap(model, VALID_JSON, IMAGES_DIR, max_images=eval_cap)
        line += f" | val_loss {vl:.4f} | mAP {ap['mAP']:.4f} | mAP50 {ap['mAP50']:.4f}"
    print(line, f"  (saved -> {OUT_DIR})")

print("done ->", OUT_DIR)

## Outputs

* `/kaggle/working/checkpoints/mask2former-nanostar/` — a plain Mask2Former
  checkpoint (model + processor), saved **every epoch**. Load it anywhere with
  `Mask2FormerForUniversalSegmentation.from_pretrained(...)`, or with the repo's
  `src/m2f_pipeline.py` for full-frame inference.

Everything under `/kaggle/working` is saved as the notebook's output and is
downloadable from the **Output** tab.

### Tuning
| Variable | Default | Notes |
|---|---|---|
| `IMG_SIZE` | `1024` | drop to `768`/`512` if you hit CUDA OOM |
| `BATCH_SIZE` / `GRAD_ACCUM` | `1` / `2` | effective batch = product; raise accum, not batch, on 16 GB |
| `EPOCHS` | `20` | |
| `SAMPLES_PER_EPOCH` | `600` | random crops drawn per epoch (dataset has 2×N distinct) |
| `SMOKE_TEST` | `True` | set `False` for the real run |

**CUDA OOM:** lower `IMG_SIZE` first, then ensure `BATCH_SIZE = 1` and raise
`GRAD_ACCUM`. **Internet Off:** add the backbone as a Kaggle model/dataset and set
`MODEL` to its local path.